# Movie Recommender System

This notebook trains two matrix-factorization models on MovieLens 1M:

- **SVD** (`scipy.sparse.linalg.svds`)
- **PMF** (Probabilistic Matrix Factorization trained with SGD)

It evaluates both models, explains the recommendations, and writes every report to `reports/`.

In [1]:
import json
import os

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.metrics import root_mean_squared_error
from sklearn.model_selection import train_test_split

from models.pmf_model import train_pmf
from models.svd_model import train_svd_model
from utils.data_loader import load_movielens_data
from utils.matrix_creation import create_and_save_interaction_matrix
from utils.recommendation import (
    add_reasons, favorite_genres, generate_recommendations, save_user_recommendations
)

os.makedirs("reports", exist_ok=True)

## 1. Load the data and look at it

We check the size, the sparsity and the rating distribution. The matrix is very empty, which is why we use latent factors instead of storing every rating.

In [2]:
ratings, movies, users = load_movielens_data("data")

n_users = ratings.UserID.nunique()
n_movies = ratings.MovieID.nunique()
sparsity = 1 - len(ratings) / (n_users * n_movies)

print(f"Ratings: {len(ratings):,}")
print(f"Users: {n_users:,}")
print(f"Movies: {n_movies:,}")
print(f"Sparsity: {sparsity:.2%}")
print("\nRating distribution (%):")
print((ratings.Rating.value_counts(normalize=True).sort_index() * 100).round(2))

Ratings: 1,000,209
Users: 6,040
Movies: 3,706
Sparsity: 95.53%

Rating distribution (%):
Rating
1     5.62
2    10.75
3    26.11
4    34.89
5    22.63
Name: proportion, dtype: float64


### What the EDA tells us

- The matrix is about **95% empty**, so most user/movie pairs are unknown.
- Ratings are **positive-skewed**: 4 and 5 stars are the most common.
- Some users rate thousands of movies while others rate only a few, so a user bias is useful.
- Drama, Comedy and Action are the most popular genres, so genre overlap is a good simple explanation for a recommendation.

## 2. Build the normalized user-item matrix

We split with `random_state=42`, pivot the training ratings into a matrix, and subtract a baseline:

```text
baseline = global mean + user bias + movie bias
```

The result is saved to `processed/user_item_matrix.csv`.

In [3]:
train, test, interaction_matrix, normalized_matrix, baseline = create_and_save_interaction_matrix(ratings)

print("Train/test:", len(train), len(test))
print("Matrix shape:", normalized_matrix.shape)
print("Saved processed/user_item_matrix.csv")

fig, ax = plt.subplots(figsize=(7, 5))
image = ax.imshow(normalized_matrix.iloc[:30, :30], cmap="coolwarm", aspect="auto")
ax.set(title="Normalized residuals (first 30 users and movies)",
       xlabel="Movie index", ylabel="User index")
fig.colorbar(image, ax=ax, label="Residual rating")
fig.tight_layout()
fig.savefig("reports/matrix_sample.png")
plt.close(fig)

Train/test: 900188 100021
Matrix shape: (6040, 3706)
Saved processed/user_item_matrix.csv


## 3. Train SVD

SVD keeps the `k` strongest latent directions of the normalized matrix. Adding the baseline back gives the final rating prediction.

In [4]:
svd_predictions, svd_rmse, svd_actuals, svd_preds = train_svd_model(
    normalized_matrix, baseline, test, k=30
)
print(f"SVD RMSE: {svd_rmse:.4f}")

SVD RMSE: 0.8950


## 4. Train PMF

PMF learns one small vector per user and per movie, then uses their dot product. The training loop minimizes squared error with L2 regularization.

We also keep 10% of the training data as **validation**, so we can plot a validation curve next to the training curve. A small ensemble of 5 models (different random seeds) is averaged to make PMF more stable.

In [5]:
pmf_train, pmf_validation = train_test_split(
    train, test_size=0.1, random_state=42
)

pmf_predictions, pmf_rmse, pmf_history, pmf_val_history = train_pmf(
    pmf_train, test, normalized_matrix, validation=pmf_validation
)
print(f"PMF RMSE: {pmf_rmse:.4f}")

PMF RMSE: 0.8469


In [6]:
plt.figure(figsize=(8, 5))
plt.plot(range(1, len(pmf_history) + 1), pmf_history, label="Training MSE")
plt.plot(range(1, len(pmf_val_history) + 1), pmf_val_history, label="Validation MSE")
plt.xlabel("Epoch")
plt.ylabel("MSE")
plt.title("PMF convergence")
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.savefig("reports/pmf_convergence.png")
plt.close()

**Stopping rule and overfitting.** We train for 100 epochs. The training MSE keeps falling, but the validation MSE flattens, which shows that more epochs would mostly memorize the training data. L2 regularization keeps the factors small, so we stop at 100 epochs.

## 5. Compare the models

The bias baseline (global mean + user bias + movie bias) is a simple collaborative-filtering benchmark. Both matrix-factorization models should beat it.

In [7]:
user_index = normalized_matrix.index.get_indexer(test.UserID)
item_index = normalized_matrix.columns.get_indexer(test.MovieID)
valid = (user_index >= 0) & (item_index >= 0)

benchmark_predictions = np.clip(
    baseline[user_index[valid], item_index[valid]], 1, 5
)
benchmark_rmse = root_mean_squared_error(
    test.Rating.to_numpy()[valid], benchmark_predictions
)
print(f"Bias baseline RMSE: {benchmark_rmse:.4f}")

metrics = {
    "SVD_RMSE": round(float(svd_rmse), 4),
    "PMF_RMSE": round(float(pmf_rmse), 4),
    "PMF_vs_SVD_improvement_%": round(
        float((svd_rmse - pmf_rmse) / svd_rmse * 100), 2
    ),
    "SVD_MSE": round(float(svd_rmse ** 2), 4),
    "PMF_MSE": round(float(pmf_rmse ** 2), 4),
    "Benchmark_CF_RMSE": round(float(benchmark_rmse), 4),
}
with open("reports/model_metrics.json", "w") as file:
    json.dump(metrics, file, indent=2)
metrics

Bias baseline RMSE: 0.9376


{'SVD_RMSE': 0.895,
 'PMF_RMSE': 0.8469,
 'PMF_vs_SVD_improvement_%': 5.37,
 'SVD_MSE': 0.801,
 'PMF_MSE': 0.7173,
 'Benchmark_CF_RMSE': 0.9376}

In [8]:
plt.figure(figsize=(7, 4.5))
plt.bar(["SVD", "PMF", "Bias baseline"],
        [svd_rmse, pmf_rmse, benchmark_rmse],
        color=["#3498db", "#2ecc71", "#95a5a6"])
plt.ylabel("RMSE")
plt.title("Held-out RMSE comparison")
plt.tight_layout()
plt.savefig("reports/rmse_comparison.png")
plt.close()

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
for ax, name, actual, predicted, color in [
    (axes[0], "SVD", svd_actuals, svd_preds, "#3498db"),
    (axes[1], "PMF", test.Rating.to_numpy()[valid],
     pmf_predictions[user_index[valid], item_index[valid]], "#2ecc71"),
]:
    ax.scatter(actual[:1000], predicted[:1000], alpha=0.4, color=color)
    ax.plot([1, 5], [1, 5], "r--")
    ax.set(title=name, xlabel="Actual rating", ylabel="Predicted rating",
           xlim=(0.8, 5.2), ylim=(0.8, 5.2))
fig.tight_layout()
fig.savefig("reports/predicted_vs_actual.png")
plt.close(fig)

## 6. Interpret the latent factors

Each column of `V` is a learned movie community. The movies with the largest values share hidden taste patterns. This is the **global** interpretation of the model.

In [9]:
V = np.load("reports/pmf_factors/V.npy")
title_by_id = movies.set_index("MovieID").Title

for factor in range(3):
    top = np.argsort(V[:, factor])[-5:][::-1]
    print(f"PMF factor {factor + 1}:")
    for idx in top:
        print(f"  {title_by_id[normalized_matrix.columns[idx]]}")

PMF factor 1:
  Waking the Dead (1999)
  McCabe & Mrs. Miller (1971)
  Naked (1993)
  Careful (1992)
  Nashville (1975)
PMF factor 2:
  Niagara (1953)
  Velvet Goldmine (1998)
  Plan 9 from Outer Space (1958)
  Star Trek V: The Final Frontier (1989)
  Faraway, So Close (In Weiter Ferne, So Nah!) (1993)
PMF factor 3:
  Thomas and the Magic Railroad (2000)
  King and I, The (1999)
  Mirror Has Two Faces, The (1996)
  Kid, The (2000)
  Prince of Tides, The (1991)


## 7. Recommendations for 3 users

We pick two users from the training set and one from the test set. For every user we save the top-10 SVD and PMF recommendations with a reason (genre match or latent factors).

In [10]:
train_counts = train.UserID.value_counts()
test_counts = test.UserID.value_counts()

train_users = list(train_counts.head(2).index)
test_user = next(user for user in test_counts.index if user not in train_users)
sample_users = train_users + [test_user]

print("Two training users:", train_users)
print("One test user:", test_user)

Two training users: [4169, 1680]
One test user: 1941


In [11]:
for user_id in sample_users:
    genres = favorite_genres(user_id, ratings, movies)
    all_recs = []
    for name, predictions in [("SVD", svd_predictions), ("PMF", pmf_predictions)]:
        recs = generate_recommendations(
            user_id, (predictions, normalized_matrix, movies, ratings)
        )
        recs = add_reasons(recs, genres).assign(Model=name)
        all_recs.append(recs)

    combined = pd.concat(all_recs)
    save_user_recommendations(user_id, combined)
    print(f"\nUser {user_id} top genres: {sorted(genres)}")
    print(combined[["Model", "Title", "PredictedRating", "Reason"]].to_string(index=False))


User 4169 top genres: ['Comedy', 'Drama', 'Thriller']
Model                                                             Title  PredictedRating                        Reason
  SVD                       South Park: Bigger, Longer and Uncut (1999)         5.000000           Genre match: Comedy
  SVD                                Gate of Heavenly Peace, The (1995)         4.966697      High latent-factor score
  SVD                                           Ulysses (Ulisse) (1954)         4.966532      High latent-factor score
  SVD                                                  Baby, The (1973)         4.966501      High latent-factor score
  SVD                                          Bittersweet Motel (2000)         4.965971      High latent-factor score
  SVD                                          One Little Indian (1973)         4.964904    Genre match: Comedy, Drama
  SVD Institute Benjamenta, or This Dream People Call Human Life (1995)         4.964319            Genre match:


User 1680 top genres: ['Comedy', 'Drama', 'Thriller']
Model                                                             Title  PredictedRating                     Reason
  SVD                               I Am Cuba (Soy Cuba/Ya Kuba) (1964)         4.978783         Genre match: Drama
  SVD                                    When Harry Met Sally... (1989)         4.978562        Genre match: Comedy
  SVD Institute Benjamenta, or This Dream People Call Human Life (1995)         4.977507         Genre match: Drama
  SVD                                          One Little Indian (1973)         4.974356 Genre match: Comedy, Drama
  SVD                                           Ulysses (Ulisse) (1954)         4.973209   High latent-factor score
  SVD                                                      Lured (1947)         4.972847   High latent-factor score
  SVD                         Schlafes Bruder (Brother of Sleep) (1995)         4.972145         Genre match: Drama
  SVD            

In [12]:
def plot_recommendations(user_id, ax):
    genres = favorite_genres(user_id, ratings, movies)
    recs = generate_recommendations(
        user_id, (pmf_predictions, normalized_matrix, movies, ratings)
    )
    matches = recs.Genres.apply(
        lambda value: bool(set(str(value).split("|")) & genres)
    )
    colors = matches.map({True: "#2ecc71", False: "#95a5a6"})
    ax.barh(recs.Title, recs.PredictedRating, color=colors)
    ax.invert_yaxis()
    ax.set(title=f"User {user_id}", xlabel="Predicted rating",
           ylabel="Movie", xlim=(0, 5.5))


fig, axes = plt.subplots(1, len(sample_users), figsize=(18, 5))
for ax, user_id in zip(axes, sample_users):
    plot_recommendations(user_id, ax)
fig.suptitle("PMF top-10 recommendations (green = favorite genre)")
fig.tight_layout()
fig.savefig("reports/top_recommendations.png")
plt.close(fig)

## 8. SVD vs PMF for the same user

We plot the two prediction rows for one user and count how many top-10 movies they share.

In [13]:
user_id = sample_users[0]
user_row = normalized_matrix.index.get_loc(user_id)

plt.figure(figsize=(10, 4))
plt.plot(svd_predictions[user_row, :50], label="SVD")
plt.plot(pmf_predictions[user_row, :50], label="PMF")
plt.title(f"SVD vs PMF predictions for user {user_id}")
plt.xlabel("Movie index (first 50)")
plt.ylabel("Predicted rating")
plt.legend()
plt.tight_layout()
plt.savefig("reports/user_comparison.png")
plt.close()

svd_ids = set(generate_recommendations(
    user_id, (svd_predictions, normalized_matrix, movies, ratings)
).MovieID)
pmf_ids = set(generate_recommendations(
    user_id, (pmf_predictions, normalized_matrix, movies, ratings)
).MovieID)
print(f"The models share {len(svd_ids & pmf_ids)}/10 recommendations for user {user_id}.")

The models share 0/10 recommendations for user 4169.


### How the models differ

SVD keeps only the strongest directions of the centered matrix, so its top scores are often pushed to 5.0. PMF spreads predictions more smoothly. That is why the two models can share very few top-10 movies for the same user. The next cells compare their per-user accuracy on the test set.

In [14]:
eval_data = test.loc[valid].copy()
eval_data["SVD_Pred"] = svd_predictions[user_index[valid], item_index[valid]]
eval_data["PMF_Pred"] = pmf_predictions[user_index[valid], item_index[valid]]
eval_data["SVD_SqError"] = (eval_data.Rating - eval_data.SVD_Pred) ** 2
eval_data["PMF_SqError"] = (eval_data.Rating - eval_data.PMF_Pred) ** 2

user_rmse = eval_data.groupby("UserID").agg(
    n=("Rating", "size"),
    SVD_RMSE=("SVD_SqError", lambda values: np.sqrt(values.mean())),
    PMF_RMSE=("PMF_SqError", lambda values: np.sqrt(values.mean())),
)

more_accurate, less_accurate = train_users
print(f"User {more_accurate}: PMF RMSE={user_rmse.loc[more_accurate, 'PMF_RMSE']:.4f}")
print(f"User {less_accurate}: PMF RMSE={user_rmse.loc[less_accurate, 'PMF_RMSE']:.4f}")
print("The less accurate user has a noisier history, so the latent factors fit them less closely.")

profiles = []
for user_id in train_users:
    history = ratings[ratings.UserID == user_id].merge(movies, on="MovieID")
    profiles.append({
        "UserID": user_id,
        "ratings": len(history),
        "mean_rating": history.Rating.mean(),
        "std_rating": history.Rating.std(),
        "genres": history.Genres.str.split("|").explode().nunique(),
    })
pd.DataFrame(profiles).set_index("UserID").round(2)

User 4169: PMF RMSE=0.7810
User 1680: PMF RMSE=1.0586
The less accurate user has a noisier history, so the latent factors fit them less closely.


,ratings,mean_rating,std_rating,genres
UserID,,,,
4169,2314,3.55,1.03,18
1680,1850,3.56,1.17,18


### Why is one training user more accurate?

The table shows the two training users. The more accurate user usually has **more ratings** and a **lower rating standard deviation** (more consistent taste). The less accurate user has a noisier or more unusual history, so the latent factors cannot fit their test ratings as closely. This is a local example of the accuracy difference.

## Summary

- The matrix is about 95% empty, so latent factors are needed.
- SVD and PMF both beat the bias baseline.
- PMF uses L2 regularization and a 5-model ensemble; train/validation curves justify stopping at 100 epochs.
- Recommendations exclude movies the user already rated and include a short reason.
- All required files are written to `processed/` and `reports/`.